**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Online Learning & Regret

The theory that unifies this curriculum's two halves: [adaptive filtering](./Intro_AdFilt_APA.ipynb) *is* online learning, and **regret** — how much worse you did than the best fixed strategy chosen in hindsight — is the guarantee those workshops never stated. Three sessions, ending with LMS getting the theorem it always deserved. Guarantees hold even when the data is *adversarial*: no statistics required.

## 1. Pre-requisites

- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) (convexity, gradients).
- [APA workshop](./Intro_AdFilt_APA.ipynb) — the algorithms about to receive theory.
- [Reinforcement Learning](../Intro_Mach_Learn/Reinforcement_Learning.ipynb) S1 — bandits are online learning with *partial* feedback.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Prediction with Experts & Hedge* (~40 min)
**Goal:** beat the best expert in hindsight — even against an adversary — via multiplicative weights.
**Feeds into:** Session 2 (online gradient descent).

---

## 2. The Experts Problem

💡 **Intuition.** Each day, $N$ 'experts' make predictions; you must combine them; then losses are revealed — possibly chosen by an **adversary who read your algorithm**. You cannot always be right, but you can guarantee to nearly match the best single expert *in hindsight*. **Hedge** does it with multiplicative weights: $w_i \mathrel{*}= e^{-\eta \ell_i}$ — exponentially discredit whoever erred. The guarantee $\text{Regret} \le \sqrt{T \ln N / 2} \cdot 2$-ish is *distribution-free*: no probability assumptions anywhere, a totally different kind of promise than [estimation theory's](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

In [ ]:
# Hedge vs an ADVERSARIAL loss sequence (designed to punish any single expert)
# adversary: each expert is good in its own recurring 'season', terrible otherwise

# YOUR CODE HERE


**What just happened.** Final regret **1.6** against a proven bound of **52.7**, with per-round excess loss of 0.0008 and falling. The `assert` passed, so the theorem held — and the curve stays well under the dashed $\sqrt{T \ln N / 2}$ line throughout.

**But read this result carefully, because the headline number flatters us.** Look at how the loss matrix is built: `losses[t, (t // 125) % N_exp] = 0.0` over 2000 rounds with 125-round seasons gives exactly 16 seasons for 16 experts, so *every* expert is perfect for one season and useless for the other fifteen. By symmetry all sixteen finish with identical cumulative loss, $2000 - 125 = 1875$. The "best expert in hindsight" is therefore no better than the worst, and even a fixed uniform weighting would have paid $2000 \times (1 - 1/16) = 1875$ — tying the best expert exactly, with no algorithm at all.

So regret of 1.6 does not show Hedge outperforming its bound by 33×. It shows this particular sequence has nothing to learn: there is no good expert to concentrate on, and matching the field is trivial. The bound is loose here because it is a *worst-case* guarantee, and this instance is far from the worst case — not because the analysis is weak.

**What the demo does legitimately establish** is the shape of the promise. Regret grows like $\sqrt{T}$ while the horizon grows like $T$, so the *average* excess loss goes to zero: whatever the sequence, per-round performance converges to the best fixed expert's. That is the meaningful statement, and the $0.0008 \to 0$ figure is the one to point at.

To see Hedge actually working, break the symmetry — give one expert a low loss every round (`losses[:, 3] = 0.2`) and re-run. The weights then concentrate on that expert within a few hundred rounds and the regret curve reflects a genuine identification problem. Comparing the two runs is the clearest way to see that regret measures *relative* performance, and that a small number can mean "we did well" or merely "there was nothing to be gained."

---
### 🕐 Session 2 of 3 — *Online Gradient Descent & the Regret Bound* (~40 min)
**Goal:** prove the O(√T) regret of OGD — the cleanest nontrivial proof in machine learning.
**Builds on:** Session 1; [Optimization](../Intro_Math/Optimization/Optimization.ipynb). &nbsp; **Feeds into:** Session 3 (the adaptive-filtering reunion).

---

## 3. OGD and Its Guarantee

Setting: at each round pick $w_t$, adversary reveals convex loss $\ell_t$, you pay $\ell_t(w_t)$ and update $w_{t+1} = w_t - \eta \nabla \ell_t(w_t)$ (projected into a radius-$R$ ball).

**Theorem.** For convex losses with $\|\nabla \ell_t\| \le G$: with $\eta = \frac{R}{G\sqrt{T}}$,
$$\text{Regret}_T = \sum_t \ell_t(w_t) - \min_{\|u\|\le R} \sum_t \ell_t(u) \;\le\; RG\sqrt{T}.$$

**Proof** (three lines — the famous potential argument). Let $u$ be any comparator; expand the 'distance potential':
$$\|w_{t+1} - u\|^2 \le \|w_t - \eta \nabla_t - u\|^2 = \|w_t - u\|^2 - 2\eta \nabla_t^T(w_t - u) + \eta^2\|\nabla_t\|^2$$
(projection only shrinks distance). Convexity gives $\ell_t(w_t) - \ell_t(u) \le \nabla_t^T (w_t - u)$; substitute and sum over $t$ — the potential terms *telescope*:
$$\text{Regret}_T \le \frac{\|w_1 - u\|^2}{2\eta} + \frac{\eta}{2}\sum_t \|\nabla_t\|^2 \le \frac{R^2}{2\eta} + \frac{\eta G^2 T}{2}.$$
Optimize $\eta$ → $RG\sqrt{T}$. $\blacksquare$ No statistics, no stationarity — the data may be chosen by a demon and the bound still holds.

In [ ]:
# Watch the theorem hold on adversarial online linear regression
# best FIXED comparator in hindsight (least squares over the whole stream, norm-capped)

# YOUR CODE HERE


**What just happened.** Final regret **−1251** against a bound of $RG\sqrt{T} = 657$. The theorem is satisfied, comfortably — and the reason the numbers look strange is worth being precise about.

**The bound is one-sided.** It promises regret $\le RG\sqrt{T}$. It says nothing whatsoever about a floor. Negative regret is not a violation, a bug, or a fluke: it means OGD accumulated *less* total loss than the best single fixed $w$ chosen with full hindsight. Students routinely misread the inequality as a two-sided prediction, so it is worth writing $-1251 \le 657$ on the board and letting the arithmetic settle it.

**Why it happened here.** The world switches targets twice, cycling through three unrelated $u$ vectors. Any *fixed* comparator must serve all three regimes with one weight vector, so `lstsq` returns a compromise that is mediocre everywhere — good for none of the three. OGD is under no such constraint: after each switch it simply re-converges over the following few hundred rounds and spends most of its time well fitted to whatever regime is current. Adaptivity beats commitment when the environment moves, and the negative regret is that advantage measured.

**The honest reading of this is a limitation, not a victory.** Regret against the best fixed strategy is a *weak benchmark* in a non-stationary world — so weak that a mediocre adaptive algorithm can beat it. The right response is not to celebrate the negative number but to notice the yardstick has stopped being informative. The literature's answer is **dynamic regret**, which compares against the best *sequence* $u_1, \dots, u_T$ rather than a single fixed $u$, with bounds that degrade gracefully as a function of how much the comparator moves. That is the framework you actually want for tracking, and it is what Session 3's adaptive-filtering reunion is implicitly reaching for.

**What remains genuinely impressive.** Nothing in the proof assumed the data was stationary, Gaussian, independent, or generated by any process at all — the targets could have been chosen by an adversary reading this code, and the $RG\sqrt{T}$ ceiling would still hold. That is a categorically different kind of promise from the statistical guarantees in [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb), which are sharper but evaporate the moment their assumptions fail. Here the assumptions are convexity and a bounded gradient, and that is all.

---
### 🕐 Session 3 of 3 — *The Adaptive-Filtering Reunion* (~35 min)
**Goal:** LMS = OGD on squared loss: restate the filtering workshops as regret guarantees.
**Builds on:** Session 2; [APA](./Intro_AdFilt_APA.ipynb).

---

## 4. LMS Gets Its Theorem

💡 **Intuition.** Look again at OGD on the squared loss $\ell_t(w) = \tfrac12(d_t - w^T x_t)^2$: the update is $w \mathrel{+}= \eta \, e_t x_t$ — **that is LMS, symbol for symbol**. So the entire regret machinery transfers: LMS is guaranteed to track within $O(\sqrt{T})$ of the best fixed filter *chosen after seeing all the data*, with no stationarity, no Gaussian noise, no eigenvalue conditions — a guarantee the [Wiener-theory story](../Intro_DSP/Statistical_Signal_Processing.ipynb) cannot make when its assumptions fail. The two views are complementary: statistics gives *sharper* answers when its assumptions hold; regret gives *unbreakable* ones when they don't. (NLMS ≈ per-round step normalization; [RLS](./Intro_RLS.ipynb) ≈ online Newton and can achieve $O(\log T)$ regret on strongly-convex losses; tracking *drifting* systems is 'dynamic regret'.)

In [ ]:
# The same system-ID scenario as the APA workshop — but scored by REGRET, and with a
# mid-stream system SWITCH that violates every stationarity assumption.

# YOUR CODE HERE


**What just happened.** The line `w = w + eta * e * xv` is LMS exactly as the [adaptive filtering workshops](./Intro_AdFilt_APA.ipynb) wrote it — and it is also online gradient descent on the squared loss, symbol for symbol, since $\nabla_w \tfrac12(d - w^\top x)^2 = -e\,x$. Nothing was reimplemented for this session. The same code now simply gets scored by a different yardstick.

That reframing is the point of the workshop. LMS was introduced as a heuristic: a stochastic approximation to gradient descent on a cost surface, whose analysis required stationarity, independence assumptions, and eigenvalue conditions on the input autocorrelation. Session 2's three-line proof hands that identical algorithm a guarantee requiring none of it — only convexity and a bounded gradient. LMS always had a theorem; it was being analysed with the wrong tools.

**The switch is what makes the demo bite.** Halfway through, the system flips to $-w_{\text{true}}$, violating stationarity as completely as it can be violated. Classical Wiener analysis has nothing to say about this stream — there is no single optimal filter to converge to. The regret guarantee is untouched, because it never assumed one existed.

**Average regret came out at −5.5, and again the negative sign is about the benchmark.** No fixed filter can serve a stream that inverts halfway: the hindsight-optimal $u^\star$ from `lstsq` averages the two regimes into something close to useless for both. LMS tracks each half and beats that compromise outright. So the number confirms adaptivity is valuable here — and simultaneously shows that "best fixed filter" has stopped being a meaningful comparison. The guarantee (average regret $\le O(1/\sqrt{T})$ above zero) holds and is not the interesting part; the interesting part is that the yardstick has gone slack, which is precisely the motivation for **dynamic regret**, where the comparator is allowed to move.

**One caution on what is being promised.** The bound concerns cumulative *prediction loss* relative to a fixed comparator. It does not claim the weights converge to $w_{\text{true}}$, and after the switch they demonstrably chase a moving target. Good prediction and correct identification are different goals, and only the first is covered here.

Taken together with Sessions 1 and 2: Hedge, OGD, LMS, NLMS, and [RLS](./Intro_RLS.ipynb) are one family of online convex optimisation algorithms differing in how the update is preconditioned — RLS being the online-Newton end, which buys $O(\log T)$ regret on strongly convex losses. The signal processing half and the machine learning half of this curriculum have been studying the same subject in different notation.

## 5. Conclusion

Hedge beats hindsight's best expert against adversaries; OGD's three-line telescope gives $RG\sqrt{T}$; and LMS turns out to have been online gradient descent all along — carrying a distribution-free guarantee the classical story never mentioned. The curriculum's two halves were one subject.

---
## Where next

- [Concentration Inequalities](../Intro_Math/Concentration/Concentration_Inequalities.ipynb) — the statistical guarantees, for when assumptions *do* hold.
- [Reinforcement Learning](../Intro_Mach_Learn/Reinforcement_Learning.ipynb) — bandits: regret with partial feedback.
- [RLS](./Intro_RLS.ipynb) — the online-Newton end of the spectrum.